In [ ]:
import os
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'


In [ ]:
import lightkurve as lk
from lightkurve import search_lightcurve
import matplotlib.pyplot as plt
import requests
import pandas as pd
from time import perf_counter
import numpy as np
from pandarallel import pandarallel

%matplotlib inline


In [ ]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")
df = df.drop(columns=["Unnamed: 11"])
df.head()


In [ ]:
pandarallel.initialize(nb_workers=8, progress_bar=True)

def process_kic_row(kic_value):
    try:
        search = lk.search_lightcurve(
            f"KIC {kic_value}", 
            mission="Kepler", 
            author="Kepler", 
            cadence="long"
        )
        
        if len(search) == 0:
            # RETURN EMPTY ARRAYS, NOT NONE
            return pd.Series({
                "time": np.array([]), 
                "flux": np.array([]), 
                "flux_err": np.array([]), 
                "n_points": 0
            })
            
        lc = search.download_all().stitch().remove_nans()
        
        # We use np.array(...) to force casting variables
        # We use .value to get the raw numbers
        return pd.Series({
            "time": np.array(lc.time.value, dtype=float),
            "flux": np.array(lc.flux.value, dtype=float),
            "flux_err": np.array(lc.flux_err.value, dtype=float),
            "n_points": len(lc)
        })
        
    except Exception:
        # Fallback for network/timeout errors
        return pd.Series({
            "time": np.array([]), 
            "flux": np.array([]), 
            "flux_err": np.array([]), 
            "n_points": 0
        })

# Assuming your DataFrame is named 'df' and 'KIC' is the index.
# We reset the index briefly so KIC is a standard column, apply in parallel, then set it back.
df_reset = df.reset_index()
# df_reset = df.head(300).copy()

# .parallel_apply() is the magic pandarallel function that replaces .apply()
extracted_features = df_reset['KIC'].parallel_apply(process_kic_row)

# Join the newly extracted features back to the main dataframe
df_final = pd.concat([df_reset, extracted_features], axis=1).set_index('KIC')
